# 0. Environment Check


In [1]:
# ============================================================
# 0. Environment Check
# Goal:
# - Check Python version
# - Check PyTorch version
# - Check CUDA availability
# - Check GPU name
# - Check whether torch.compile is available
# ============================================================

import torch
import platform
import os
import time
import math
import statistics

# TODO 1:
# Print Python version.
# Hint: platform.python_version()
print("python", platform.python_version())


# TODO 2:
# Print PyTorch version.
# Hint: torch.__version__
print("pytorch", torch.__version__)


# TODO 3:
# Print whether CUDA is available.
# Hint: torch.cuda.is_available()
print("cuda", torch.cuda.is_available())


# TODO 4:
# If CUDA is available, print:
# - GPU name
# - CUDA version
# - GPU compute capability
# Hint:
# torch.cuda.get_device_name(0)
# torch.version.cuda
# torch.cuda.get_device_capability(0)
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0), torch.version.cuda, torch.cuda.get_device_capability(0))


# TODO 5:
# Print whether torch.compile exists.
# Hint: hasattr(torch, "compile")
print("torch.compile available:", hasattr(torch, "compile"))

python 3.12.13
pytorch 2.11.0+cu128
cuda True
gpu Tesla T4 12.8 (7, 5)
torch.compile available: True


# 1. CUDA Benchmark Utility

In [2]:
# ============================================================
# 1. CUDA Benchmark Utility
# Goal:
# - Measure first-run latency
# - Measure steady-state latency after warmup
# - Use CUDA synchronization correctly
# - Use CUDA events for GPU timing
# ============================================================

def benchmark_cuda(fn, args=(), warmup=20, iters=100, name="fn"):
    """
    Benchmark a CUDA function.

    Requirements:
    1. Measure first-run latency using time.time().
    2. Synchronize before and after timing.
    3. Run warmup iterations.
    4. Use CUDA events to measure steady-state latency.
    5. Return a dict containing:
       - name
       - first_run_ms
       - steady_avg_ms
       - steady_p50_ms
       - steady_p95_ms
    """

    # TODO 1:
    # Synchronize CUDA before first run.
    # Hint: torch.cuda.synchronize()
    torch.cuda.synchronize()


    # TODO 2:
    # Record CPU wall-clock time before first run.
    # Hint: t0 = time.time()
    t0 = time.time()


    # TODO 3:
    # Run fn(*args) once.
    # This first run may include torch.compile overhead.
    out = fn(*args)


    # TODO 4:
    # Synchronize CUDA after first run.
    torch.cuda.synchronize()


    # TODO 5:
    # Record CPU wall-clock time after first run.
    # Compute first_run_ms.
    t1 = time.time()
    first_run_ms = (t1 - t0) * 1000


    # TODO 6:
    # Warmup loop.
    # Run fn(*args) warmup times.
    # Then synchronize CUDA.
    for _ in range (warmup):
         out = fn(*args)
    torch.cuda.synchronize()



    # TODO 7:
    # Create an empty list called times.
    # For each iteration:
    #   - create start/end CUDA events
    #   - record start
    #   - run fn(*args)
    #   - record end
    #   - synchronize
    #   - append elapsed time in ms
    #
    # Hint:
    # start = torch.cuda.Event(enable_timing=True)
    # end = torch.cuda.Event(enable_timing=True)
    # start.record()
    # out = fn(*args)
    # end.record()
    # torch.cuda.synchronize()
    # times.append(start.elapsed_time(end))
    times = []
    for _ in range (iters):
         start = torch.cuda.Event(enable_timing=True)
         end = torch.cuda.Event(enable_timing=True)

         start.record()
         out = fn(*args)
         end.record()

         torch.cuda.synchronize()
         times.append(start.elapsed_time(end))


    # TODO 8:
    # Compute average, p50, p95 latency.
    # Hint:
    # statistics.mean(times)
    # statistics.median(times)
    # sorted(times)[int(0.95 * len(times)) - 1]
    avg_ms = statistics.mean(times)
    p50_ms = statistics.median(times)
    p95_ms = sorted(times)[int(0.95 * len(times)) - 1]


    # TODO 9:
    # Print result in a readable format.
    print(f"\n{name}")
    print(f"  first_run_ms: {first_run_ms:.3f}")
    print(f"  steady_avg_ms: {avg_ms:.3f}")
    print(f"  steady_p50_ms: {p50_ms:.3f}")
    print(f"  steady_p95_ms: {p95_ms:.3f}")


    # TODO 10:
    # Return result dict.
   #  pass
    return {"name": name,
            "first_run_ms": first_run_ms,
            "steady_avg_ms": avg_ms,
            "steady_p50_ms": p50_ms,
            "steady_p95_ms": p95_ms,
           }

# 2. MLP: Eager vs torch.compile

In [3]:
# ============================================================
# 2. MLP Benchmark
# Goal:
# - Compare PyTorch eager vs torch.compile on MLP
# - Observe cold-start compile overhead
# - Observe steady-state speedup
# - Understand that GEMM-heavy workload may not always get huge speedup
# ============================================================

import torch.nn as nn
import torch.nn.functional as F

class TinyMLP(nn.Module):
    def __init__(self, d_model=768, d_ff=3072):
        super().__init__()

        # TODO 1:
        # Define first Linear layer: d_model -> d_ff
        self.fc1 = nn.Linear(d_model, d_ff)



        # TODO 2:
        # Define second Linear layer: d_ff -> d_model
        self.fc2 = nn.Linear(d_ff, d_model)



    def forward(self, x):
        # TODO 3:
        # Implement:
        # x -> fc1 -> GELU -> fc2
        #
        # Hint:
        # return self.fc2(F.gelu(self.fc1(x)))
        return self.fc2(F.gelu(self.fc1(x)))
        # pass

In [4]:
# ============================================================
# MLP benchmark runner
# ============================================================

device = "cuda"
dtype = torch.float16

def run_mlp_case(B, D=768, DFF=3072, dtype=torch.float16):
    """
    Run one MLP benchmark case.

    Requirements:
    1. Create TinyMLP.
    2. Move model to CUDA and dtype.
    3. Create random input x with shape [B, D].
    4. Create eager model.
    5. Create compiled model with torch.compile(..., backend="inductor").
    6. Benchmark both.
    7. Compute steady-state speedup.
    8. Return eager result, compiled result, speedup.
    """

    print("=" * 80)
    print(f"MLP case: B={B}, D={D}, DFF={DFF}, dtype={dtype}")

    # TODO 1:
    # Create model and move it to device/dtype.
    # Set eval mode.
    model = TinyMLP(D, DFF).to(device=device, dtype=dtype).eval()



    # TODO 2:
    # Create input x.
    # Shape: [B, D]
    x = torch.randn(B, D, device=device, dtype=dtype)


    # TODO 3:
    # Set eager_model.
    eager_model = model


    # TODO 4:
    # Create compiled_model.
    # Hint: torch.compile(model, backend="inductor")
    compiled_model = torch.compile(model, backend="inductor")


    # TODO 5:
    # Use torch.no_grad().
    # Benchmark eager_model.
    with torch.no_grad():
        eager_res = benchmark_cuda(
            eager_model,
            args=(x,),
            warmup=20,
            iters=100,
            name=f"MLP eager B={B}"
        )


    # TODO 6:
    # Benchmark compiled_model.
        compiled_res = benchmark_cuda(
            compiled_model,
            args=(x,),
            warmup=20,
            iters=100,
            name=f"MLP compile B={B}"
        )

    # TODO 7:
    # Compute speedup:
    # eager steady avg / compile steady avg
    speedup = eager_res["steady_avg_ms"] / compiled_res["steady_avg_ms"]
    print(f"  steady-state speedup: {speedup:.2f}x")


    # TODO 8:
    # Print speedup and return results.
    # pass
    return eager_res, compiled_res, speedup

In [5]:
# ============================================================
# Run MLP cases
# ============================================================

mlp_results = []

# TODO:
# Run run_mlp_case for several batch sizes.
# Suggested B values:
# [1, 8, 32, 128, 512]
#
# Store results in mlp_results.
mlp_results = []

for B in [1, 8, 32, 128, 512]:
    res = run_mlp_case(B)
    mlp_results.append((B, res))


MLP case: B=1, D=768, DFF=3072, dtype=torch.float16

MLP eager B=1
  first_run_ms: 353.261
  steady_avg_ms: 0.139
  steady_p50_ms: 0.133
  steady_p95_ms: 0.170

MLP compile B=1
  first_run_ms: 7990.820
  steady_avg_ms: 0.225
  steady_p50_ms: 0.215
  steady_p95_ms: 0.287
  steady-state speedup: 0.62x
MLP case: B=8, D=768, DFF=3072, dtype=torch.float16

MLP eager B=8
  first_run_ms: 13.564
  steady_avg_ms: 0.130
  steady_p50_ms: 0.125
  steady_p95_ms: 0.157

MLP compile B=8
  first_run_ms: 1031.389
  steady_avg_ms: 0.349
  steady_p50_ms: 0.366
  steady_p95_ms: 0.439
  steady-state speedup: 0.37x
MLP case: B=32, D=768, DFF=3072, dtype=torch.float16

MLP eager B=32
  first_run_ms: 2.947
  steady_avg_ms: 0.148
  steady_p50_ms: 0.141
  steady_p95_ms: 0.187

MLP compile B=32
  first_run_ms: 0.479
  steady_avg_ms: 0.273
  steady_p50_ms: 0.261
  steady_p95_ms: 0.419
  steady-state speedup: 0.54x
MLP case: B=128, D=768, DFF=3072, dtype=torch.float16

MLP eager B=128
  first_run_ms: 36.446
  stea

torch.compile is not guaranteed to speed up every model. For GEMM-dominated workloads, eager PyTorch may already dispatch to highly optimized vendor libraries such as cuBLAS, so the compiler has limited room to improve. The biggest benefits often come from fusion-heavy, elementwise-heavy, reduction-heavy, or Python-overhead-heavy workloads.

# 3. Elementwise Fusion Experiment

In [6]:
# ============================================================
# 3. Elementwise Fusion
# Goal:
# - Create a chain of elementwise operations
# - Compare eager vs torch.compile
# - Observe fusion benefits
# ============================================================

def elementwise_chain(x):
    """
    Implement the following chain:

    y = x + 1.0
    y = y * 2.0
    y = sigmoid(y)
    y = y + relu(x)
    return y

    This is a good case for TorchInductor fusion.
    """

    # TODO 1:
    # y = x + 1.0
    y = x + 1.0


    # TODO 2:
    # y = y * 2.0
    y = y * 2.0


    # TODO 3:
    # y = torch.sigmoid(y)
    y = torch.sigmoid(y)


    # TODO 4:
    # y = y + torch.relu(x)
    y = y + torch.relu(x)


    # TODO 5:
    # return y
    # pass
    return y


# TODO 6:
# Compile elementwise_chain with torch.compile backend="inductor".
compiled_elementwise_chain = torch.compile(elementwise_chain, backend="inductor")

In [7]:
def run_elementwise_case(N, dtype=torch.float16):
    """
    Run one elementwise benchmark case.

    Requirements:
    1. Create x with shape [N].
    2. Benchmark eager elementwise_chain.
    3. Benchmark compiled elementwise_chain.
    4. Compute speedup.
    """

    print("=" * 80)
    print(f"Elementwise case: N={N}, dtype={dtype}")

    # TODO 1:
    # Create random x on CUDA with shape [N].
    x = torch.randn(N, device="cuda", dtype=dtype)


    # TODO 2:
    # Benchmark eager function.
    with torch.no_grad():
        eager_res = benchmark_cuda(
            elementwise_chain,
            args=(x,),
            warmup=20,
            iters=100,
            name=f"Elementwise eager N={N}"
        )


    # TODO 3:
    # Benchmark compiled function.
    compiled_res = benchmark_cuda(
        compiled_elementwise_chain,
        args=(x,),
        warmup=20,
        iters=100,
        name=f"Elementwise compile N={N}"
    )


    # TODO 4:
    # Compute speedup.
    speedup = eager_res["steady_avg_ms"] / compiled_res["steady_avg_ms"]


    # TODO 5:
    # Print and return results.
    # pass
    print(f"  steady-state speedup: {speedup:.2f}x")
    return eager_res, compiled_res, speedup


In [8]:
# ============================================================
# Run elementwise cases
# ============================================================

elementwise_results = []

# TODO:
# Run run_elementwise_case for:
# 1024
# 1024 * 1024
# 16 * 1024 * 1024
for N in [1024, 1024 * 1024, 16 * 1024 * 1024]:
    elementwise_results.append((N, run_elementwise_case(N)))

Elementwise case: N=1024, dtype=torch.float16

Elementwise eager N=1024
  first_run_ms: 106.434
  steady_avg_ms: 0.075
  steady_p50_ms: 0.071
  steady_p95_ms: 0.092

Elementwise compile N=1024
  first_run_ms: 432.811
  steady_avg_ms: 0.101
  steady_p50_ms: 0.094
  steady_p95_ms: 0.128
  steady-state speedup: 0.74x
Elementwise case: N=1048576, dtype=torch.float16

Elementwise eager N=1048576
  first_run_ms: 0.193
  steady_avg_ms: 0.102
  steady_p50_ms: 0.100
  steady_p95_ms: 0.109

Elementwise compile N=1048576
  first_run_ms: 425.305
  steady_avg_ms: 0.119
  steady_p50_ms: 0.114
  steady_p95_ms: 0.145
  steady-state speedup: 0.86x
Elementwise case: N=16777216, dtype=torch.float16

Elementwise eager N=16777216
  first_run_ms: 1.750
  steady_avg_ms: 1.535
  steady_p50_ms: 1.537
  steady_p95_ms: 1.552

Elementwise compile N=16777216
  first_run_ms: 0.623
  steady_avg_ms: 0.479
  steady_p50_ms: 0.463
  steady_p95_ms: 0.538
  steady-state speedup: 3.20x


### Elementwise Fusion Benchmark

The elementwise chain benchmark shows a clear workload-size-dependent behavior. For small tensors, `torch.compile` was slower than eager execution because kernel launch overhead, wrapper overhead, and measurement noise dominate the runtime. However, for a large tensor with 16,777,216 elements, the compiled version achieved a 3.20× steady-state speedup.

This result indicates that TorchInductor can effectively fuse multiple elementwise operations into a smaller number of kernels, reducing intermediate tensor materialization and global memory traffic. The speedup becomes significant when the workload is large enough for memory bandwidth to dominate.

# 4. Attention: Naive vs Compile vs SDPA

In [9]:
# ============================================================
# 4. Attention Benchmark
# Goal:
# - Compare naive attention eager
# - Compare naive attention compiled
# - Compare PyTorch SDPA
# - Compare compiled SDPA
#
# Key idea:
# torch.compile can optimize the naive graph,
# but specialized attention kernels may still be faster.
# ============================================================

def naive_attention(q, k, v):
    """
    Implement standard scaled dot-product attention manually.

    Input shapes:
    q, k, v: [B, H, N, D]

    Steps:
    1. scores = q @ k.transpose(-2, -1) / sqrt(D)
    2. probs = softmax(scores, dim=-1)
    3. out = probs @ v
    4. return out
    """

    # TODO 1:
    # Get head dimension D from q.
    d = q.size(-1)

    # TODO 2:
    # Compute attention scores.
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d)

    # TODO 3:
    # Apply softmax over last dimension.
    probs = torch.softmax(scores, dim=-1)

    # TODO 4:
    # Multiply probs with v.
    out = torch.matmul(probs, v)

    # TODO 5:
    # Return output.
    # pass
    return out


def sdpa_attention(q, k, v):
    """
    Use PyTorch scaled_dot_product_attention.

    Hint:
    F.scaled_dot_product_attention(q, k, v, is_causal=False)
    """

    # TODO:
    # Return SDPA output.
    # pass
    return F.scaled_dot_product_attention(q, k, v, is_causal=False)


# TODO:
# Compile naive_attention and sdpa_attention using torch.compile backend="inductor".
compiled_naive_attention = torch.compile(naive_attention, backend="inductor")
compiled_sdpa_attention = torch.compile(sdpa_attention, backend="inductor")

In [10]:
def run_attention_case(B=1, H=8, N=512, D=64, dtype=torch.float16):
    """
    Run one attention benchmark case.

    Requirements:
    1. Create random q, k, v with shape [B, H, N, D].
    2. Benchmark naive_attention eager.
    3. Benchmark compiled naive_attention.
    4. Benchmark sdpa_attention eager.
    5. Benchmark compiled sdpa_attention.
    6. Return all results.
    """

    print("=" * 80)
    print(f"Attention case: B={B}, H={H}, N={N}, D={D}, dtype={dtype}")

    # TODO 1:
    # Create q, k, v on CUDA.
    q = torch.randn(B, H, N, D, device="cuda", dtype=dtype)
    k = torch.randn(B, H, N, D, device="cuda", dtype=dtype)
    v = torch.randn(B, H, N, D, device="cuda", dtype=dtype)


    # TODO 2:
    # Benchmark naive_attention eager.
    eager_naive = benchmark_cuda(
        naive_attention,
        args=(q, k, v),
        warmup=20,
        iters=50,
        name=f"Naive attention eager N={N}"
    )


    # TODO 3:
    # Benchmark compiled naive_attention.
    compile_naive = benchmark_cuda(
        compiled_naive_attention,
        args=(q, k, v),
        warmup=20,
        iters=50,
        name=f"Naive attention compile N={N}"
    )


    # TODO 4:
    # Benchmark sdpa_attention eager.
    eager_sdpa = benchmark_cuda(
        sdpa_attention,
        args=(q, k, v),
        warmup=20,
        iters=50,
        name=f"SDPA eager N={N}"
    )


    # TODO 5:
    # Benchmark compiled sdpa_attention.
    compile_sdpa = benchmark_cuda(
        compiled_sdpa_attention,
        args=(q, k, v),
        warmup=20,
        iters=50,
        name=f"SDPA compile N={N}"
    )

    # TODO 6:
    # Return all results.
    return eager_naive, compile_naive, eager_sdpa, compile_sdpa
    # pass

In [11]:
# ============================================================
# Run attention cases
# ============================================================

attention_results = []

# TODO:
# Run run_attention_case for sequence lengths:
# [128, 256, 512, 1024]
for N in [128, 256, 512, 1024]:
    attention_results.append((N, run_attention_case(N=N)))

Attention case: B=1, H=8, N=128, D=64, dtype=torch.float16

Naive attention eager N=128
  first_run_ms: 73.269
  steady_avg_ms: 0.130
  steady_p50_ms: 0.124
  steady_p95_ms: 0.156

Naive attention compile N=128
  first_run_ms: 299.220
  steady_avg_ms: 0.125
  steady_p50_ms: 0.123
  steady_p95_ms: 0.142

SDPA eager N=128
  first_run_ms: 1.833
  steady_avg_ms: 0.063
  steady_p50_ms: 0.053
  steady_p95_ms: 0.106

SDPA compile N=128
  first_run_ms: 142.567
  steady_avg_ms: 0.116
  steady_p50_ms: 0.111
  steady_p95_ms: 0.143
Attention case: B=1, H=8, N=256, D=64, dtype=torch.float16

Naive attention eager N=256
  first_run_ms: 1.093
  steady_avg_ms: 0.128
  steady_p50_ms: 0.123
  steady_p95_ms: 0.143

Naive attention compile N=256
  first_run_ms: 764.139
  steady_avg_ms: 0.265
  steady_p50_ms: 0.273
  steady_p95_ms: 0.305

SDPA eager N=256
  first_run_ms: 0.117
  steady_avg_ms: 0.072
  steady_p50_ms: 0.070
  steady_p95_ms: 0.082

SDPA compile N=256
  first_run_ms: 207.478
  steady_avg_ms: 0

### Attention Benchmark Observation

I compared naive PyTorch attention, compiled naive attention, SDPA, and compiled SDPA across sequence lengths from 128 to 1024.

The compiled naive attention only showed a clear improvement at larger sequence length. At N=1024, `torch.compile` reduced naive attention latency from 0.689 ms to 0.484 ms, achieving about 1.42× speedup. This suggests that TorchInductor can optimize parts of the naive attention graph when the workload is large enough.

However, PyTorch SDPA eager execution was consistently faster than both naive eager and naive compiled attention. At N=1024, SDPA eager achieved 0.366 ms, compared with 0.484 ms for compiled naive attention. This indicates that specialized attention kernels are more effective than general graph compilation for attention workloads.

Compiling SDPA itself did not improve performance and was slower than SDPA eager in this setup, likely because SDPA already dispatches to optimized backend kernels and the additional compile wrapper provides limited extra optimization.

# 5. Graph Break Experiment

In [ ]:
# ============================================================
# 5. Graph Break Experiment
# Goal:
# - Use torch._dynamo.explain
# - Observe graph breaks caused by .item(), Python control flow, print, side effects
# ============================================================

import torch._dynamo as dynamo

def show_explain(fn, *args):
    """
    Print TorchDynamo explanation.

    Requirements:
    1. Print function name.
    2. Run dynamo.explain(fn)(*args).
    3. Print explanation.
    """

    print("=" * 80)
    print(f"Explain for: {fn.__name__}")

    # TODO 1:
    # Call dynamo.explain(fn)(*args)


    # TODO 2:
    # Print explanation
    pass

In [ ]:
# ============================================================
# Case 1: pure tensor ops, should have few/no graph breaks
# ============================================================

def no_break_fn(x):
    """
    Implement:
    return torch.relu(x + 1.0) * 2.0
    """

    # TODO:
    pass


x = torch.randn(1024, device="cuda")

# TODO:
# Run show_explain(no_break_fn, x)


In [ ]:
# ============================================================
# Case 2: .item() + Python if
# This often causes graph break because tensor value escapes to Python.
# ============================================================

def item_break_fn(x):
    """
    Implement:

    if x.sum().item() > 0:
        return x * 2.0
    else:
        return x - 2.0
    """

    # TODO:
    pass


# TODO:
# Run show_explain(item_break_fn, x)


In [ ]:
# ============================================================
# Case 3: Rewrite with torch.where
# This converts Python control flow into tensor operation.
# ============================================================

def where_fixed_fn(x):
    """
    Implement:

    cond = x.sum() > 0
    return torch.where(cond, x * 2.0, x - 2.0)
    """

    # TODO:
    pass


# TODO:
# Run show_explain(where_fixed_fn, x)


In [ ]:
# ============================================================
# Case 4: print/logging inside function
# Python-side I/O may interfere with graph capture.
# ============================================================

def print_break_fn(x):
    """
    Implement:

    print("shape:", x.shape)
    return x.relu()
    """

    # TODO:
    pass


# TODO:
# Run show_explain(print_break_fn, x)


In [ ]:
# ============================================================
# Case 5: Python side effect
# Mutating Python state inside compiled region is usually bad.
# ============================================================

cache = []

def side_effect_fn(x):
    """
    Implement:

    cache.append(x)
    return x + 1
    """

    # TODO:
    pass


# TODO:
# Run show_explain(side_effect_fn, x)


# 6. Dynamic Shape / Recompile Experiment

In [ ]:
# ============================================================
# 6. Dynamic Shape Experiment
# Goal:
# - Compare dynamic=False vs dynamic=True
# - Vary batch size and sequence length
# - Observe first-time shape cost and repeated-shape behavior
# ============================================================

def simple_transformer_like(x, w1, w2):
    """
    Input:
    x:  [B, N, D]
    w1: [D, DFF]
    w2: [DFF, D]

    Implement:
    y = x @ w1
    y = GELU(y)
    y = y @ w2
    return y
    """

    # TODO 1:
    # y = torch.matmul(x, w1)


    # TODO 2:
    # y = F.gelu(y)


    # TODO 3:
    # y = torch.matmul(y, w2)


    # TODO 4:
    # return y
    pass

In [ ]:
# ============================================================
# Prepare weights and compiled functions
# ============================================================

D = 256
DFF = 1024
dtype = torch.float16

# TODO 1:
# Create w1 with shape [D, DFF] on CUDA.


# TODO 2:
# Create w2 with shape [DFF, D] on CUDA.


# TODO 3:
# Compile simple_transformer_like with dynamic=False.


# TODO 4:
# Compile simple_transformer_like with dynamic=True.


In [ ]:
def run_shape_sequence(fn, shapes, name):
    """
    Run the same compiled function with different input shapes.

    Requirements:
    1. For each shape [B, N, D]:
       - create x
       - synchronize
       - measure one run with time.time()
       - synchronize
       - print elapsed time
    2. Return list of (shape, elapsed_ms)
    """

    print("=" * 80)
    print(name)

    results = []

    for shape in shapes:
        B, N, D_local = shape

        # TODO 1:
        # Create x with shape [B, N, D_local].


        # TODO 2:
        # Synchronize and record start time.


        # TODO 3:
        # Run fn(x, w1, w2).


        # TODO 4:
        # Synchronize and record end time.


        # TODO 5:
        # Compute elapsed_ms.


        # TODO 6:
        # Print shape and elapsed_ms.


        # TODO 7:
        # Append result.

        pass

    return results

In [ ]:
# ============================================================
# Vary batch size
# ============================================================

batch_shapes = [
    (1, 128, D),
    (2, 128, D),
    (4, 128, D),
    (8, 128, D),
    (1, 128, D),
]

# TODO:
# Run batch_shapes with compiled_static.


# TODO:
# Run batch_shapes with compiled_dynamic.


In [ ]:
# ============================================================
# Vary sequence length
# ============================================================

seq_shapes = [
    (1, 64, D),
    (1, 128, D),
    (1, 256, D),
    (1, 512, D),
    (1, 128, D),
]

# TODO:
# Run seq_shapes with compiled_static.


# TODO:
# Run seq_shapes with compiled_dynamic.


# 7. Inspect Inductor Generated Code

In [ ]:
# ============================================================
# 7. Inspect TorchInductor Generated Code
# Goal:
# - Use TORCH_LOGS="output_code"
# - Observe generated code
# - Look for Triton kernels or extern kernels
# ============================================================

%%writefile inspect_inductor_skeleton.py
import torch
import torch.nn.functional as F

def elementwise_chain(x):
    """
    TODO:
    Implement the same elementwise chain:

    y = x + 1.0
    y = y * 2.0
    y = torch.sigmoid(y)
    y = y + torch.relu(x)
    return y
    """

    # TODO:
    pass


# TODO 1:
# Create x with shape [1024 * 1024] on CUDA, dtype fp16.


# TODO 2:
# Compile elementwise_chain with torch.compile backend="inductor".


# TODO 3:
# Run compiled function several times.


# TODO 4:
# Synchronize CUDA.


print("done")

In [ ]:
# TODO:
# Run the script with TORCH_LOGS="output_code"
# Hint:
# !TORCH_LOGS="output_code" python inspect_inductor_skeleton.py


In [ ]:
# TODO:
# Try TORCH_COMPILE_DEBUG=1
# Hint:
# !TORCH_COMPILE_DEBUG=1 python inspect_inductor_skeleton.py


# 8. Backend Comparison

In [ ]:
# ============================================================
# 8. Backend Comparison
# Goal:
# - Compare raw eager
# - torch.compile backend="eager"
# - torch.compile backend="aot_eager"
# - torch.compile backend="inductor"
#
# Meaning:
# backend="eager": useful for Dynamo capture debugging
# backend="aot_eager": useful for AOTAutograd/debugging
# backend="inductor": real optimizing backend
# ============================================================

def train_like_fn(x, w):
    """
    Implement:

    y = x @ w
    y = GELU(y)
    loss = y.square().mean()
    return loss
    """

    # TODO:
    pass


B, D, H = 128, 512, 2048

# TODO 1:
# Create x with shape [B, D] on CUDA, dtype fp16.


# TODO 2:
# Create w with shape [D, H] on CUDA, dtype fp16.


# TODO 3:
# Compile train_like_fn with backend="eager".


# TODO 4:
# Compile train_like_fn with backend="aot_eager".


# TODO 5:
# Compile train_like_fn with backend="inductor".


# TODO 6:
# Benchmark raw eager.


# TODO 7:
# Benchmark backend="eager".


# TODO 8:
# Benchmark backend="aot_eager".


# TODO 9:
# Benchmark backend="inductor".


# 9. fullgraph=True Debugging

In [ ]:
# ============================================================
# 9. fullgraph=True Debugging
# Goal:
# - See that fullgraph=True turns graph breaks into explicit failures
# ============================================================

def bad_fn(x):
    """
    Implement a function that likely breaks graph:

    if x.sum().item() > 0:
        return x * 2
    return x - 2
    """

    # TODO:
    pass


try:
    # TODO 1:
    # Compile bad_fn with fullgraph=True.


    # TODO 2:
    # Run compiled_bad on a CUDA tensor.


    # TODO 3:
    # Synchronize CUDA.


    print("bad_fn compiled successfully")
except Exception as e:
    print("Compilation failed with fullgraph=True")
    print(type(e))
    print(e)

In [ ]:
def good_fn(x):
    """
    Implement graph-friendly version:

    cond = x.sum() > 0
    return torch.where(cond, x * 2, x - 2)
    """

    # TODO:
    pass


# TODO 1:
# Compile good_fn with fullgraph=True.


# TODO 2:
# Run compiled_good on CUDA tensor.


# TODO 3:
# Synchronize CUDA.


print("good_fn compiled successfully")